# 🎵 ACE-Step 1.5 QUALITÉ + cache Google Drive — MusicGen Studio

Mode qualité (modèle **sft** 50 étapes + **planification par le modèle de langage**), avec une copie de l'installation et des modèles sur **Google Drive** pour relancer vite dans une nouvelle session.

1. Menu **Exécution → Modifier le type d'exécution → T4 GPU** → **Enregistrer**.
2. Clique sur **▶** ci-dessous.
3. Une fenêtre Google s'ouvre : choisis le compte **bastaps@gmail.com** (celui qui a l'espace Drive) et autorise l'accès.
4. Attends le **cadre vert** et copie le lien `https://….gradio.live` dans **MusicGen Studio → ⚙ Réglages → Lien ACE-Step**, moteur **Colab GPU**.

- **1re fois** : installation complète (~15-20 min), puis sauvegarde automatique sur Drive en arrière-plan (~10 min, le studio est déjà utilisable).
- **Sessions suivantes** : restauration depuis Drive, lien vert en quelques minutes.
- Dossier sur Drive : `MusicGen_Colab_cache/` (~15 Go). Le supprimer = repartir de zéro.

Laisse l'onglet ouvert pendant que tu utilises le studio.


In [ ]:
#@title ▶ Lancer ACE-Step 1.5 QUALITÉ + cache Drive (clique sur le bouton ▶ à gauche)
#@markdown Laisse les options par défaut. Tout est automatique : Drive, installation ou restauration, modèles, puis lien vert.
import os, re, subprocess, sys, time, collections, threading, shutil
from IPython.display import HTML, display

COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"   # version d'ACE-Step 1.5 testée : ne pas changer
REPO = "/content/ACE-Step-1.5"
CACHE = "/content/drive/MyDrive/MusicGen_Colab_cache"
VENV_TAR = f"{CACHE}/ace15_{COMMIT[:8]}_env.tar"      # environnement Python (uv + .venv)
CKPT_TAR = f"{CACHE}/ace15_checkpoints.tar"           # modèles téléchargés (sft, LM, VAE…)
UV_DIRS = ["root/.local/bin/uv", "root/.local/bin/uvx", "root/.local/share/uv/python"]
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

def box(msg, ok=True):
    bg, bd = ("#0f3d2a", "#2ecc71") if ok else ("#4a1414", "#e74c3c")
    display(HTML(f'<div style="margin:10px 0;padding:14px 18px;border-radius:10px;background:{bg};'
                 f'border:2px solid {bd};color:#fff;font-family:sans-serif;font-size:15px">{msg}</div>'))

def stop(msg):
    box("❌ " + msg, ok=False)
    raise SystemExit(msg)

def sh(cmd, cwd=None, tries=1):
    for i in range(tries):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
        if r.returncode == 0:
            return r.stdout
        if i < tries - 1:
            print(f"   nouvel essai ({i + 2}/{tries})…"); time.sleep(10)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
    stop("Étape échouée : " + cmd.split()[0] + ". Envoie une capture de ce message.")

def ok_file(tar):
    return os.path.exists(tar) and os.path.exists(tar + ".ok")

def restore(tar, what):
    t0 = time.time()
    r = subprocess.run(f"tar -xf '{tar}' -C /", shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   ⚠ restauration {what} impossible ({r.stderr[-200:]}) : installation normale")
        return False
    print(f"   ♻ {what} restauré depuis Drive en {int(time.time() - t0)} s")
    return True

def save(paths, tar, what):
    """Archive (sans compression : plus rapide) écrite sous un nom temporaire puis renommée : jamais d'archive à moitié écrite."""
    paths = [p for p in paths if os.path.exists("/" + p)]
    tmp = tar + ".part"
    r = subprocess.run(["tar", "-cf", tmp, "-C", "/"] + paths, capture_output=True, text=True)
    if r.returncode == 0:
        os.replace(tmp, tar)
        open(tar + ".ok", "w").write(time.strftime("%Y-%m-%d %H:%M"))
        print(f"\n💾 {what} sauvegardé sur Drive ({os.path.getsize(tar) // 2**20} Mo) : la prochaine session sera rapide.")
    else:
        print(f"\n⚠ Sauvegarde {what} sur Drive impossible : {r.stderr[-200:]}")
        try: os.remove(tmp)
        except OSError: pass

# 1) GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    stop("Pas de GPU. Menu <b>Exécution → Modifier le type d'exécution → T4 GPU</b> → Enregistrer, puis relance ▶.")
print("✅ 1/5 GPU :", gpu)

# 2) Google Drive (cache) : facultatif, le notebook marche aussi sans
drive_ok = os.path.isdir("/content/drive/MyDrive")
if not drive_ok:
    print("⏳ 2/5 Connexion à Google Drive : choisis le compte bastaps@gmail.com dans la fenêtre Google…")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_ok = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("   ⚠ Drive non connecté (" + str(e)[:120] + ") : on continue sans cache.")
if drive_ok:
    os.makedirs(CACHE, exist_ok=True)
    print("✅ 2/5 Google Drive connecté — cache :", "présent" if ok_file(VENV_TAR) else "vide (1re fois)")
else:
    print("✅ 2/5 Sans Google Drive (pas de cache)")

# 3) Code (version figée)
if not os.path.isdir(REPO + "/.git"):
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/ACE-Step/ACE-Step-1.5.git")
    sh(f"git fetch -q --depth 1 origin {COMMIT} && git checkout -q FETCH_HEAD", cwd=REPO, tries=3)
print("✅ 3/5 Code ACE-Step 1.5 prêt")

# 4) Dépendances : déjà là (même session) > copie Drive > installation complète
fresh_env = False
if not os.path.exists(REPO + "/.venv/bin/acestep"):
    if drive_ok and ok_file(VENV_TAR):
        print("⏳ 4/5 Restauration de l'installation depuis Drive (1 à 3 min)…")
        restore(VENV_TAR, "environnement")
        # copie d'une ancienne session : si Colab a changé de Python entre-temps, elle ne démarre plus -> réinstallation
        chk = subprocess.run([REPO + "/.venv/bin/python", "-c", "import torch, acestep"], capture_output=True, text=True)
        if chk.returncode != 0:
            print("   ⚠ copie Drive incompatible avec cette session Colab : réinstallation propre")
            shutil.rmtree(REPO + "/.venv", ignore_errors=True)
    if not os.path.exists(REPO + "/.venv/bin/acestep"):
        print("⏳ 4/5 Installation des dépendances (5 à 10 min la première fois)…")
        if not shutil.which("uv"):
            sh("curl -LsSf https://astral.sh/uv/install.sh | sh", tries=3)
        sh("uv sync --frozen", cwd=REPO, tries=3)
        fresh_env = True
if not os.path.isdir(REPO + "/checkpoints") and drive_ok and ok_file(CKPT_TAR):
    print("⏳    Restauration des modèles depuis Drive (2 à 5 min)…")
    restore(CKPT_TAR, "modèles")
print("✅ 4/5 Dépendances installées")

# Correctif T4 : float16 donne des sons vides (NaN) sur les GPU anciens -> float32
orch = REPO + "/acestep/core/generation/handler/init_service_orchestrator.py"
src = open(orch, encoding="utf-8").read()
new = re.sub(r"(else:\n\s+self\.dtype = torch\.)float16(\n\s+logger\.info\(\n\s+\"\[initialize_service\] Pre-Ampere)",
             r"\1float32\2", src)
if new != src:
    open(orch, "w", encoding="utf-8").write(new)
elif "self.dtype = torch.float32\n                    logger.info(" not in src:
    stop("Correctif float32 introuvable.")

# 5) Serveur + lien
subprocess.run("pkill -f acestep_v15_pipeline; pkill -f '.venv/bin/acestep'", shell=True)
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND")}
env.update(PYTHONUNBUFFERED="1", GRADIO_ANALYTICS_ENABLED="False", MPLBACKEND="Agg")
cmd = [REPO + "/.venv/bin/acestep", "--port", "7860", "--share", "--server-name", "127.0.0.1",
       "--init_service", "true", "--config_path", "acestep-v15-sft",
       "--init_llm", "true", "--lm_model_path", "acestep-5Hz-lm-0.6B", "--backend", "pt",
       "--offload_to_cpu", "true", "--quantization", "none"]
print("⏳ 5/5 Démarrage du serveur (chargement des modèles ; 1re fois : téléchargement 5 à 10 min)…")
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)

def backup():
    """Après le lien vert : copie sur Drive de ce qui manque encore (une seule fois)."""
    if not drive_ok:
        return
    if fresh_env or not ok_file(VENV_TAR):
        save(["content/ACE-Step-1.5/.venv"] + UV_DIRS, VENV_TAR, "Environnement")
    if os.path.isdir(REPO + "/checkpoints") and not ok_file(CKPT_TAR):
        save(["content/ACE-Step-1.5/checkpoints"], CKPT_TAR, "Modèles")

tail = collections.deque(maxlen=60)
shown = False
KEEP = ("Error", "ERROR", "Traceback", "out of memory", "NaN")
try:
    for line in proc.stdout:
        tail.append(line.rstrip())
        m = re.search(r"https://[a-z0-9-]+\.gradio\.live", line)
        if m and not shown:
            shown = True
            box(f'✅ <b>ACE-Step 1.5 QUALITÉ est prêt.</b> Copie ce lien dans <b>MusicGen Studio → ⚙ Réglages → Lien ACE-Step</b> :'
                f'<div style="font-size:22px;font-weight:700;margin-top:8px;user-select:all;word-break:break-all">{m.group(0)}</div>'
                f'<div style="font-size:12px;margin-top:8px;opacity:.8">Le lien marche tant que cette cellule tourne '
                f'(icône qui tourne). Ne ferme pas cet onglet.</div>')
            if drive_ok and (fresh_env or not ok_file(VENV_TAR) or not ok_file(CKPT_TAR)):
                print("💾 Sauvegarde sur Drive en arrière-plan (environ 10 min, une seule fois) — tu peux déjà utiliser le studio.")
                threading.Thread(target=backup, daemon=True).start()
        elif any(k in line for k in KEEP):
            print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("⏹ Serveur arrêté.")

code = proc.wait()
if code not in (0, None, -15):
    print("\n".join(list(tail)[-30:]))
    box(f"⚠️ Le serveur s'est arrêté (code {code}). Envoie une capture de ce message, "
        "ou relance ▶ pour obtenir un nouveau lien.", ok=False)
